In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

import pickle
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [8]:
with open("dataset/dataset.pkl", "rb") as f:
    dataset = pickle.load(f)
    x_files = dataset["gtr"]
    y_files = dataset["ney"]

In [9]:
x_train_files, x_test_files, y_train_files, y_test_files = train_test_split(
    x_files, y_files, test_size=0.2, random_state=42)

print(len(x_train_files), len(x_test_files))

200 50


In [10]:
print(x_train_files[0], y_train_files[0])
print(x_test_files[0], y_test_files[0])

gtr_132.npy ney_132.npy
gtr_142.npy ney_142.npy


In [11]:
NEY_SPECTROGRAM_DIR = "dataset/spectrograms/ney/"
GTR_SPECTROGRAM_DIR = "dataset/spectrograms/ac_gtr/"

In [12]:
class SpectrogramDataset(Dataset):
    def __init__(self,
                 x_files,
                 y_files,
                 ney_spectrogram_dir,
                 gtr_spectrogram_dir):
        self.x_files = x_files
        self.y_files = y_files
        self.ney_spectrogram_dir = ney_spectrogram_dir
        self.gtr_spectrogram_dir = gtr_spectrogram_dir
        self.x = []
        self.y = []

        # gtr files
        for f in self.x_files:
            spectrogram = np.load(
                self.gtr_spectrogram_dir + f, allow_pickle=True)
            self.x.append(spectrogram)

        # ney files
        for f in self.y_files:
            spectrogram = np.load(
                self.ney_spectrogram_dir + f, allow_pickle=True)
            self.y.append(spectrogram)

        # prepare gtr spectrograms
        self.x = np.array(self.x)
        # min max scaling
        self.x = (self.x - np.min(self.x)) / (np.max(self.x) - np.min(self.x))
        # expand dims to imitate grayscale img format
        self.x = np.expand_dims(self.x, axis=3)
        # re-arrange dims for CNN
        self.x = np.transpose(self.x, (0, 3, 2, 1))
        
        # prepare ney spectrograms
        self.y = np.array(self.y)
        # min max scaling
        self.y = (self.y - np.min(self.y)) / (np.max(self.y) - np.min(self.y))
        # expand dims to imitate grayscale img format
        self.y = np.expand_dims(self.y, axis=3)
        # re-arrange dims for CNN
        self.y = np.transpose(self.y, (0, 3, 2, 1))

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.x.shape[0]

In [13]:
train_dataset = SpectrogramDataset(x_files,
                                   y_files,
                                   NEY_SPECTROGRAM_DIR,
                                   GTR_SPECTROGRAM_DIR)

In [14]:
train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    shuffle=True)

In [17]:
x, y = next(iter(train_data_loader))

In [18]:
x.shape

torch.Size([16, 1, 40, 256])

In [19]:
print("Original shape:", x.shape)
x = nn.Conv2d(1, 16, 3, stride=2, padding=1)(x)
print("After Conv2d(16, 16, 3, stride=2, padding=1):", x.shape)
# x = nn.Conv2d(16, 32, 3, stride=2, padding=1)(x)
# print("After Conv2d(16, 32, 3, stride=2, padding=1):", x.shape)
# x = nn.Conv2d(32, 64, 7)(x)
# print("After Conv2d(32, 64, 7):", x.shape)
# x = nn.ConvTranspose2d(64, 32, 7)(x)
# print("After ConvTranspose2d(64, 32, 7):", x.shape)
# x = nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1)(x)
# print("After ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1):", x.shape)
# x = nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1)(x)
# print("After ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1):", x.shape)

Original shape: torch.Size([16, 1, 40, 256])
After Conv2d(16, 16, 3, stride=2, padding=1): torch.Size([16, 16, 20, 128])


In [ ]:
class AutoEncoderCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 7)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 7),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded